# EXTRAÇÃO DATALAKE - ZGLMM159 - Centro 4058

In [0]:
%sql
-- =====================================================================
-- PASSO 1 - EXTRAÇÃO DATALAKE - ZGLMM159 - Centro 4058
-- Tabela: dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
-- Objetivo: gerar 1 arquivo único para comparar com o consolidado SAP
-- =====================================================================


-- ---------------------------------------------------------------------
-- 1.1 CONFERÊNCIA RÁPIDA (rode antes de exportar)
-- Se retornar 0, o centro pode estar gravado com outro formato
-- ---------------------------------------------------------------------
SELECT COUNT(*) AS total_linhas_4058
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058';

-- Se der 0, teste variações de preenchimento do centro:
-- SELECT DISTINCT cod_centro
-- FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
-- WHERE cod_centro LIKE '%4058%';


# EXTRAÇÃO PRINCIPAL

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 1.2 EXTRAÇÃO PRINCIPAL - todas as 36 colunas do centro 4058
-- Aliases já no padrão dos cabeçalhos das suas planilhas SAP
-- ---------------------------------------------------------------------
SELECT
    -- ===== CHAVES =====
      TRIM(cod_material)                             AS Material
    , TRIM(cod_centro)                               AS Centro
    , TRIM(cod_deposito)                             AS Deposito
    , TRIM(cod_empresa)                              AS Empresa

    -- ===== C1 - CONTÁBIL / VALORAÇÃO =====
    , TRIM(cod_classe_avaliacao)                     AS Classe_avaliacao
    , TRIM(tp_apropriacao_ledger_materiais)          AS Determ_preco
    , TRIM(ind_controle_preco)                       AS Controle_preco
    , qt_unidade_preco                               AS Unidade_preco
    , vl_preco_medio_movel                           AS PMM
    , TRIM(tp_avaliacao)                             AS Tipo_avaliacao

    -- ===== M1 - MRP 1 / TAMANHO DE LOTE =====
    , TRIM(tp_grupo_mrp)                             AS Grupo_MRP
    , TRIM(cod_regra_calculo_tamanho_lotes_planejamento) AS RegraCalcTamLotes
    , TRIM(tp_mrp)                                   AS Tipo_de_MRP
    , TRIM(cod_planejador_mrp)                       AS Planejador_MRP
    , TRIM(cod_horizonte_planejamento_fixo)          AS Hor_plan_fix
    , TRIM(cod_abc)                                  AS Codigo_ABC
    , TRIM(tp_grupo_unidade_medida)                  AS Grupo_UM
    , TRIM(cod_status_material_centro)               AS St_mat_esp_cen
    , TRIM(cod_perfil_arredondamento)                AS Perfil_arred
    , qt_tamanho_fixo_lote                           AS Tam_fixo_lote
    , qt_tamanho_maximo_lote                         AS Tam_maximo_lote
    , qt_tamanho_minimo_lote                         AS Tam_min_lote
    , qt_valor_arredondamento_pedida                 AS ValArredond
    , qt_estoque_maximo                              AS Estoque_maximo
    , qt_ponto_reabastecimento                       AS Ponto_reabast
    , TRIM(dt_inicio_validade_status)                AS Valido_desde

    -- ===== M2 - MRP 2 / 3 - SUPRIMENTO E PROGRAMAÇÃO =====
    , TRIM(tp_suprimento)                            AS Recrutamento
    , TRIM(tp_grupo_determinacao_estoque)            AS Grupo_determ_estoque
    , TRIM(ind_coproduto)                            AS Coproduto
    , TRIM(cod_local_inventario_producao)            AS Deposito_prod
    , TRIM(ind_codigo_material_granel)               AS Material_granel
    , TRIM(tp_suprimento_especial)                   AS Suprmto_espec
    , vl_tempo_producao_interna                      AS Tempo_prod_int
    , qt_estoque_seguranca                           AS Estq_seguranca
    , qt_dias_prazo_entrega_previsto                 AS Prz_entrg_prev
    , qt_dias_processamento_entrada                  AS Tempo_proc_EM

FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058'
ORDER BY Material, Deposito;


# ESTATÍSTICAS DA EXTRAÇÃO

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 1.3 ESTATÍSTICAS DA EXTRAÇÃO (anote os números para o relatório)
-- ---------------------------------------------------------------------
SELECT
      COUNT(*)                                            AS total_linhas
    , COUNT(DISTINCT cod_material)                        AS materiais_distintos
    , COUNT(DISTINCT cod_deposito)                        AS depositos_distintos
    , COUNT(DISTINCT cod_empresa)                         AS empresas_distintas
    , COUNT(DISTINCT CONCAT(cod_material,'|',cod_deposito)) AS chaves_mat_deposito
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058';



# CHECAGEM DE DUPLICIDADE NA CHAVE

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 1.4 CHECAGEM DE DUPLICIDADE NA CHAVE (Material + Depósito)
-- Se retornar linhas, a chave não é única e o Python vai precisar tratar
-- ---------------------------------------------------------------------
SELECT
      cod_material
    , cod_deposito
    , COUNT(*) AS qtd_repeticoes
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058'
GROUP BY cod_material, cod_deposito
HAVING COUNT(*) > 1
ORDER BY qtd_repeticoes DESC
LIMIT 50;



# FORMATO DA DATA

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 1.5 FORMATO DA DATA (crítico para o passo 3)
-- No Datalake é STRING; no Excel do SAP veio como MM/DD/AAAA
-- ---------------------------------------------------------------------
SELECT DISTINCT
      dt_inicio_validade_status        AS exemplo
    , LENGTH(dt_inicio_validade_status) AS tamanho
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058'
  AND dt_inicio_validade_status IS NOT NULL
  AND TRIM(dt_inicio_validade_status) <> ''
ORDER BY tamanho
LIMIT 20;


In [0]:
%sql
-- Verifica se tp_avaliacao sempre vem vazio e traz exemplos quando preenchido
SELECT
    TRIM(cod_material) AS Material,
    TRIM(cod_centro) AS Centro,
    TRIM(cod_deposito) AS Deposito,
    TRIM(tp_avaliacao) AS Tipo_avaliacao
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE tp_avaliacao IS NOT NULL
  AND TRIM(tp_avaliacao) <> ''
LIMIT 20;


In [0]:
%sql
-- 1) Confirma que a tabela tem 1 linha por Material+Centro
SELECT
    COUNT(*)                                        AS total_linhas,
    COUNT(DISTINCT cod_material)                    AS materiais_distintos,
    COUNT(DISTINCT CONCAT(cod_material,'|',cod_centro)) AS material_centro_distintos
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058';

In [0]:
%sql
-- 2) Mostra o que o Datalake gravou para os materiais de exemplo
SELECT
    TRIM(cod_material) AS Material,
    TRIM(cod_centro) AS Centro,
    TRIM(cod_deposito) AS Deposito,
    TRIM(tp_grupo_mrp) AS Grupo_MRP,
    TRIM(tp_mrp) AS Tipo_de_MRP,
    TRIM(cod_planejador_mrp) AS Planejador_MRP,
    TRIM(cod_abc) AS Codigo_ABC,
    qt_estoque_maximo AS Estoque_maximo,
    qt_ponto_reabastecimento AS Ponto_reabast,
    TRIM(dt_inicio_validade_status) AS Valido_desde
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058'
  AND CAST(CAST(cod_material AS BIGINT) AS STRING) IN (
      '425263','472650','472555','171893','498167',
      '641978','628338','102724','310110','315551',
      '334277','343106'
  )
ORDER BY cod_material;

In [0]:
%sql
-- 3) Procura QUALQUER material com mais de uma linha (deveria retornar só o 260762)
SELECT
    cod_material,
    cod_centro,
    COUNT(*)                          AS qtd_linhas,
    COUNT(DISTINCT cod_deposito)      AS qtd_depositos,
    COLLECT_SET(cod_deposito)         AS depositos
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058'
GROUP BY cod_material, cod_centro
HAVING COUNT(*) > 1
ORDER BY qtd_linhas DESC;

In [0]:
%sql
-- 4) Distribuição de depósitos gravados (visão geral)
SELECT
    CASE WHEN cod_deposito IS NULL OR TRIM(cod_deposito) IN ('','null')
         THEN 'VAZIO' ELSE cod_deposito END AS deposito,
    COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058'
GROUP BY 1
ORDER BY qtd DESC;

In [0]:
%sql

SELECT
    TRIM(cod_material) AS Material,
    TRIM(cod_centro) AS Centro,
    TRIM(tp_grupo_mrp) AS Grupo_MRP
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_centro = '4058'
  AND cod_material = '000000000000567863';


In [0]:
%sql
